In [1]:
# ===== SETUP DE PATH =====
# Adiciona o diretório pai ao path para garantir que 'source' seja encontrado
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

print(f"✅ Project root adicionado ao sys.path: {project_root}")
print(f"   Notebook directory: {notebook_dir}")

# ===== AUTORELOAD =====
# Carrega automaticamente mudanças nos módulos
%load_ext autoreload
%autoreload 2

print("✅ Autoreload configurado (autoreload 2)")

# ===== IMPORTS DO source/ =====
from source.config import (
    PROJ_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR,
    get_data_path
)
from loguru import logger

logger.info(f"📁 Projeto: {PROJ_ROOT}")

✅ Project root adicionado ao sys.path: c:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering
   Notebook directory: c:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\notebooks
✅ Autoreload configurado (autoreload 2)
2026-03-14 11:56:07.793 | INFO     | source.config:<module>:77 - [CONFIG] PROJ_ROOT: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering
2026-03-14 11:56:07.814 | INFO     | source.config:<module>:78 - [CONFIG] Python executando de: c:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\notebooks
2026-03-14 11:56:07.814 | INFO     | __main__:<module>:28 - 📁 Projeto: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering


# indice

- [1 Load libs](#1-Load-libs)
- [2 Config path](#2-Config-path)


## 1 Load libs

In [2]:
import pandas as pd

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)

# Configuração para exibir todas as linhas
pd.set_option('display.max_rows', None)

# Configuração para que o conteúdo de uma coluna não seja cortado
pd.set_option('display.max_colwidth', None)

# Configuração para expandir a largura da exibição para que mais colunas caibam na tela
pd.set_option('display.width', 1000)


## 2 Config path

In [3]:
# Caminhos dinâmicos usando config.py (não hardcoded)
path_accounts_df = get_data_path("HI-Medium_accounts.csv", "external")
path_trans_df = get_data_path("HI-Medium_Trans.csv", "external")

logger.info(f"Accounts: {path_accounts_df}")
logger.info(f"Trans: {path_trans_df}")

2026-03-14 11:56:08.543 | INFO     | __main__:<module>:5 - Accounts: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\external\HI-Medium_accounts.csv
2026-03-14 11:56:08.543 | INFO     | __main__:<module>:6 - Trans: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\external\HI-Medium_Trans.csv


## 3 Union datasets

In [4]:


# Carregar os arquivos
accounts_df = pd.read_csv(path_accounts_df)
trans_df = pd.read_csv(path_trans_df)

# Renomear colunas duplicadas em trans_df para maior clareza
trans_df.columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

# 1. Juntar transações com informações da conta de origem (remetente)
trans_enriched_df = pd.merge(
    trans_df,
    accounts_df,
    left_on=['From Bank', 'From Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left'
)

# Renomear colunas para evitar conflitos
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'From Bank Name',
    'Entity ID': 'From Entity ID',
    'Entity Name': 'From Entity Name'
})

# 2. Juntar o resultado com informações da conta de destino (destinatário)
trans_enriched_df = pd.merge(
    trans_enriched_df,
    accounts_df,
    left_on=['To Bank', 'To Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left',
    suffixes=('', '_To')
)

# Renomear colunas para maior clareza
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'To Bank Name',
    'Entity ID': 'To Entity ID',
    'Entity Name': 'To Entity Name'
})

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
print(trans_enriched_df.head())


Tabela de Transações Enriquecida:
          Timestamp  From Bank From Account  To Bank To Account  Amount Received Receiving Currency  Amount Paid Payment Currency Payment Format  Is Laundering                From Bank Name  Bank ID Account Number From Entity ID        From Entity Name                  To Bank Name  Bank ID_To Account Number_To To Entity ID  To Entity Name
0  2022/09/01 00:17         20    800104D70       20  800104D70          6794.63          US Dollar      6794.63        US Dollar   Reinvestment              0  National Bank of Los Angeles       20      800104D70    2AA20590D80          Partnership #1  National Bank of Los Angeles          20         800104D70  2AA20590D80  Partnership #1
1  2022/09/01 00:02       3196    800107150     3196  800107150          7739.29          US Dollar      7739.29        US Dollar   Reinvestment              0         Acme Cooperative Bank     3196      800107150    2AA205CDDE0          Corporation #1         Acme Cooperative Bank

## 3 Salve data raw

In [5]:
# Salvar usando config.py (não hardcoded path)
output_path = get_data_path('trans_enriched.csv', 'raw')
trans_enriched_df.to_csv(output_path, index=False)

logger.success(f"✅ Dados salvos em: {output_path}")

2026-03-14 12:03:15.157 | SUCCESS  | __main__:<module>:5 - ✅ Dados salvos em: C:\Users\win\OneDrive\Área de Trabalho\TCC\money_laundering\data\raw\trans_enriched.csv


In [6]:
trans_enriched_df.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0,National Bank of Los Angeles,20,800104D70,2AA20590D80,Partnership #1,National Bank of Los Angeles,20,800104D70,2AA20590D80,Partnership #1
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0,Acme Cooperative Bank,3196,800107150,2AA205CDDE0,Corporation #1,Acme Cooperative Bank,3196,800107150,2AA205CDDE0,Corporation #1
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0,Savings Bank of the South,1208,80010E430,2AA207BA840,Partnership #2,Savings Bank of the South,1208,80010E430,2AA207BA840,Partnership #2
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0,Savings Bank of the South,1208,80010E650,2AA1FCA9B10,Sole Proprietorship #1,National Bank of Los Angeles,20,80010E6F0,2AA2080D0C0,Corporation #2
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0,Savings Bank of the South,1208,80010E650,2AA1FCA9B10,Sole Proprietorship #1,National Bank of Los Angeles,20,80010EA30,2AA206E2470,Partnership #3
